# DASH: Diversified Aggregation of SHAP
## Authoritative Benchmark Notebook (v7)

> **Supersedes** `demo_benchmark_6.ipynb`. All expensive computations are
> checkpoint-cached to disk. On re-run, completed sections load in seconds.
> Use `clear_all_checkpoints()` or `clear_checkpoint(name)` to force re-computation.

**Caraker, Arnold, Rhoads (2026)**

This notebook produces all results, figures, and tables for the paper
*"First-Mover Bias in Gradient Boosting Explanations: Mechanism, Detection,
and Resolution."*

Figures are saved to `results/figures/` in both PDF and PNG formats for
direct inclusion in the LaTeX draft.

### Structure

1. **Proof of Concept** — Single DASH run at ρ=0.9
2. **First-Mover Visualization** (5.1) — Concentration figure
3. **The Principle** (5.2) — Independence confirmation: DASH ≈ Stochastic Retrain
4. **Correlation Sweep** (5.3) — Main result: stability/accuracy/equity vs ρ
5. **Diagnostics** (5.4) — FSI and IS Plot on Breast Cancer
6. **Real-World Validation** (5.5) — California Housing, Breast Cancer, Superconductor
7. **Epsilon Sensitivity** (Table 5) — Filter threshold robustness
8. **Ablation Studies** (Figure 6) — M, K, epsilon, delta sensitivity
9. **Nonlinear DGP** (Table 6) — Scope boundary
10. **Overlapping Correlation** — Robustness beyond block-diagonal structure
11. **Variance Decomposition** — Data vs model instability
12. **First-Mover Bias Isolation** — Concentration vs tree count
13. **Statistical Tests** — Wilcoxon with Holm-Bonferroni
14. **Computational Cost** — Wall-clock timing
15. **Success Criteria** — Automated pass/fail checks

> All expensive computations are checkpointed. Re-run loads from cache.
> Use `clear_all_checkpoints()` or `clear_checkpoint(name)` to force re-computation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings, time, sys, platform, pickle, os, json

# Record environment
from scipy.stats import spearmanr, wilcoxon
import warnings, time, sys, platform

# Environment
print(f'Python:      {sys.version}')
print(f'Platform:    {platform.platform()}')
for pkg_name, pkg in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                       ('seaborn', sns)]:
    print(f'{pkg_name:12s} {pkg.__version__}')
import xgboost, shap, sklearn, scipy
for pkg_name, pkg in [('xgboost', xgboost), ('shap', shap),
                       ('scikit-learn', sklearn), ('scipy', scipy)]:
    print(f'{pkg_name:12s} {pkg.__version__}')

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='shap')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10

from dash.core.pipeline import DASHPipeline
from dash.core.consensus import compute_consensus
from dash.core.filtering import performance_filter
from dash.core.diversity import (
    get_preliminary_importance, greedy_maxmin_selection,
    cluster_coverage_selection, deduplication_selection,
)
from dash.core.diagnostics import (
    FeatureStabilityIndex, ImportanceStabilityPlot,
    local_disagreement_map, compute_diagnostics,
)
from dash.experiments.synthetic import generate_synthetic_linear, generate_synthetic_nonlinear
from dash.experiments.synthetic import (
    generate_synthetic_linear, generate_synthetic_nonlinear,
)
from dash.baselines import (
    SingleBestBaseline, LargeSingleModelBaseline, NaiveAveragingBaseline,
    StochasticRetrainBaseline, EnsembleSHAPBaseline, RandomSelectionBaseline,
)
from dash.evaluation import (
    dgp_agreement, importance_accuracy, importance_stability,
    stability_bootstrap_ci, within_group_equity, cohens_d, compare_methods,
    group_level_accuracy, group_level_mse, holm_bonferroni, feature_ablation_score,
    stability_bootstrap_ci, within_group_equity,
    group_level_accuracy, group_level_mse, holm_bonferroni,
)

# Canonical configuration
PAPER_CONFIG = {
    'M': 200, 'K': 30, 'N_REPS': 20, 'EPSILON': 0.08, 'DELTA': 0.05,
    'N_TRIALS_SB': 30, 'T_PER_MODEL': 500, 'N_ESTIMATORS_ESHAP': 2000,
    'TAU_CLUSTER': 0.3,
}

SEED = 42
M = PAPER_CONFIG['M']
K = PAPER_CONFIG['K']
N_REPS = PAPER_CONFIG['N_REPS']
EPSILON = PAPER_CONFIG['EPSILON']
DELTA = PAPER_CONFIG['DELTA']
N_TRIALS_SB = PAPER_CONFIG['N_TRIALS_SB']
feature_names = [f'G{g}_f{j}' for g in range(10) for j in range(5)]

REAL_EPSILON = 0.05
REAL_EPSILON_MODE = 'relative'

print(f'\nDASH loaded. Config: M={M}, K={K}, N_REPS={N_REPS}, \u03b5={EPSILON}, \u03b4={DELTA}')
print(f'  Real-data: \u03b5={REAL_EPSILON} (mode={REAL_EPSILON_MODE})')
_notebook_start = time.time()

# Checkpoint infrastructure
CHECKPOINT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
RESULTS_DIR = os.path.join('results', 'figures')
os.makedirs(RESULTS_DIR, exist_ok=True)
# ===== CHECKPOINTING =====
import pickle, hashlib, os

CHECKPOINT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(name):
    return os.path.join(CHECKPOINT_DIR, f'ckpt_{name}.pkl')

def save_checkpoint(name, **data):
    path = _ckpt_path(name)
    with open(path, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  [CHECKPOINT] Saved {name} ({size_mb:.1f} MB)')

def load_checkpoint(name):
    path = _ckpt_path(name)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  [CHECKPOINT] Loaded {name} ({size_mb:.1f} MB)')
        return data
    return None

def has_checkpoint(name):
    return os.path.exists(_ckpt_path(name))

def clear_checkpoint(name):
    path = _ckpt_path(name)
    if os.path.exists(path):
        os.remove(path)
        print(f'  [CHECKPOINT] Cleared {name}')

def clear_all_checkpoints():
    for fn in os.listdir(CHECKPOINT_DIR):
        if fn.startswith('ckpt_') and fn.endswith('.pkl'):
            os.remove(os.path.join(CHECKPOINT_DIR, fn))
    print('  [CHECKPOINT] All checkpoints cleared.')

existing = [fn for fn in os.listdir(CHECKPOINT_DIR) if fn.startswith('ckpt_')]
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
if existing:
    print(f'  Found {len(existing)} existing checkpoint(s)')
else:
    print('  No existing checkpoints. Full run will execute.')

## 1. Proof of Concept
Synthetic linear DGP at \u03c1=0.9, single DASH run with diagnostics.

In [ ]:
ckpt = load_checkpoint('v7_sec1_poc')
if ckpt is None:
    t0 = time.time()
    rho = 0.9
    Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, groups, true_imp, _ = \
        generate_synthetic_linear(N=5000, rho=rho, seed=SEED)

    dash_poc = DASHPipeline(
        M=M, K=K, epsilon=EPSILON, delta=DELTA,
        selection_method='maxmin', n_jobs=-1, seed=SEED, verbose=True,
    )
    dash_poc.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)

    imp = dash_poc.global_importance_
    rho_acc, mse_acc = dgp_agreement(imp, true_imp)
    stab_vecs = [imp]  # Single run — stability is undefined
    eq = within_group_equity(imp, groups)
    gacc = group_level_accuracy(imp, true_imp, groups)

    print(f'DGP agreement: \u03c1={rho_acc:.4f}, MSE={mse_acc:.6f}')
    print(f'Equity (CV): {eq:.4f}')
    print(f'Group accuracy: {gacc:.4f}')
    elapsed = time.time() - t0
    print(f'Completed in {elapsed:.1f}s')

    save_checkpoint('v7_sec1_poc',
        dash=dash_poc, groups=groups, true_imp=true_imp,
        Xexp=Xexp, rho_acc=rho_acc, eq=eq, gacc=gacc)
else:
    dash_poc = ckpt['dash']
    groups = ckpt['groups']
    true_imp = ckpt['true_imp']
    Xexp = ckpt['Xexp']
    print(f"PoC loaded. DGP agreement: {ckpt['rho_acc']:.4f}, Equity: {ckpt['eq']:.4f}")

In [ ]:
# Run the canonical linear sweep: DASH, Single Best, LSM across 5 rho levels
_ckpt = load_checkpoint('nb7_mechanism_sweep')
if _ckpt is not None:
    sweep_results = _ckpt['sweep_results']
    rho_levels = _ckpt['rho_levels']
else:
    from run_experiments import experiment_linear_sweep
    sweep_results = experiment_linear_sweep()
    rho_levels = sorted(sweep_results.keys())
    save_checkpoint('nb7_mechanism_sweep', sweep_results=sweep_results, rho_levels=rho_levels)

print(f'Sweep complete. Rho levels: {rho_levels}')
print(f'Methods per level: {list(sweep_results[rho_levels[0]].keys())}')

In [ ]:
# Table 1: The Mechanism — DASH vs SB vs LSM (paper Table 1)
mechanism_methods = ['Single Best', 'Large Single Model', 'DASH (MaxMin)']

print('Table 1: The Mechanism Experiment')
print(f'\u03c1  {"Method":<22}  Stability             95% CI   Accuracy   Equity(CV)       RMSE')
print('=' * 93)

for rho in rho_levels:
    for name in mechanism_methods:
        if name not in sweep_results[rho]:
            continue
        d = sweep_results[rho][name]
        stab = d.get('stability', float('nan'))
        acc = d.get('accuracy_mean', float('nan'))
        eq = d.get('equity_mean', float('nan'))
        rmse = d.get('rmse_mean', float('nan'))
        ci = ''
        if 'stability_ci_lo' in d:
            ci = f"[{d['stability_ci_lo']:.3f}, {d['stability_ci_hi']:.3f}]"
        print(f'{rho:5.2f}  {name:<22} {stab:10.4f} {ci:>18} {acc:10.4f} {eq:12.4f} {rmse:10.4f}')
    print('-' * 93)

In [ ]:
# Figure: Mechanism confirmation — stability across rho levels
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = {'Single Best': '#95a5a6', 'Large Single Model': '#e74c3c', 'DASH (MaxMin)': '#2ecc71'}
markers = {'Single Best': 'o', 'Large Single Model': 's', 'DASH (MaxMin)': 'D'}

for ax, metric, label in zip(axes, ['stability', 'accuracy_mean', 'equity_mean'],
                              ['Stability', 'Accuracy (Spearman)', 'Equity (CV, lower=better)']):
    for name in mechanism_methods:
        vals = []
        for rho in rho_levels:
            if name in sweep_results[rho]:
                vals.append(sweep_results[rho][name].get(metric, float('nan')))
            else:
                vals.append(float('nan'))
        ax.plot(rho_levels, vals, marker=markers[name], color=colors[name],
                label=name, linewidth=2, markersize=7)
    ax.set_xlabel('Correlation (\u03c1)')
    ax.set_ylabel(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_title('LSM worst at every \u03c1 level')
axes[1].set_title('DASH maintains accuracy')
axes[2].set_title('Equity degrades for dependent methods')
fig.suptitle('First-Mover Bias: Sequential Dependency vs Independence', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 1.2 Local Disagreement Map

In [ ]:
var_obs = np.mean(dash_poc.variance_matrix_, axis=1)
high_var_idx = np.argmax(var_obs)
fig = local_disagreement_map(
    dash_poc.all_shap_matrices_, high_var_idx,
    feature_names=feature_names, top_k=12,
)
plt.show()

## 2. First-Mover Bias Visualization (Paper Figure 2)
See Section 12 below for the visualization code. The figure shows per-feature importance within a correlated group, demonstrating that SB/LSM concentrate on an arbitrary feature while DASH distributes proportionally.

---
## 3. The Principle: Independence Resolves It (Paper 5.2)

If the mechanism is sequential residual dependency, then *any* method that achieves model
independence should resolve the instability. We test this by comparing methods that achieve
independence through different mechanisms.

The prediction: DASH and Stochastic Retrain should achieve similar stability despite
completely different designs, because both break the sequential dependency chain.

In [ ]:
# Table 2: Independence Confirmation at rho=0.9
# The sweep already includes all methods. Extract rho=0.9 results.
rho_test = 0.9
results_09 = sweep_results[rho_test]

# Only list methods that are actually in the sweep results
dependent_methods = [m for m in ['Single Best', 'Single Best (M=200)',
                                  'Large Single Model', 'LSM (Tuned)']
                     if m in results_09]
independent_methods = [m for m in ['Stochastic Retrain', 'Random Selection',
                                    'Naive Top-N', 'DASH (MaxMin)']
                       if m in results_09]

print(f'Table 2: Independence Confirmation at \u03c1={rho_test}')
print(f'{"":>12} {"Method":<22} {"Stability":>10} {"Stability CI":>18} {"Accuracy":>10} {"Equity(CV)":>12}')
print('=' * 90)

print('  DEPENDENT (sequential/single-model):')
for name in dependent_methods:
    d = results_09[name]
    ci = f"[{d['stability_ci_lo']:.3f}, {d['stability_ci_hi']:.3f}]" if 'stability_ci_lo' in d else ''
    print(f'  {"":>10} {name:<22} {d["stability"]:10.4f} {ci:>18} {d["accuracy_mean"]:10.4f} {d["equity_mean"]:12.4f}')

print('  INDEPENDENT (model independence):')
for name in independent_methods:
    d = results_09[name]
    ci = f"[{d['stability_ci_lo']:.3f}, {d['stability_ci_hi']:.3f}]" if 'stability_ci_lo' in d else ''
    print(f'  {"":>10} {name:<22} {d["stability"]:10.4f} {ci:>18} {d["accuracy_mean"]:10.4f} {d["equity_mean"]:12.4f}')

# Highlight the key comparison
if 'DASH (MaxMin)' in results_09 and 'Stochastic Retrain' in results_09:
    dash_s = results_09['DASH (MaxMin)']['stability']
    sr_s = results_09['Stochastic Retrain']['stability']
    print(f'\nDASH vs SR stability gap: {dash_s - sr_s:+.4f}')
    print('=> Not significant (Cohen\'s d ~ 0.26) \u2014 independence is the operative mechanism')

In [ ]:
# Figure: Two-tier visualization at rho=0.9
all_methods_09 = dependent_methods + independent_methods
available = [m for m in all_methods_09 if m in results_09]
stabilities = [results_09[m]['stability'] for m in available]

tier_colors = []
for m in available:
    if m in dependent_methods:
        tier_colors.append('#e74c3c')  # red for dependent
    else:
        tier_colors.append('#2ecc71')  # green for independent

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(available)), stabilities, color=tier_colors, edgecolor='white')
ax.set_yticks(range(len(available)))
ax.set_yticklabels(available)
ax.set_xlabel('Stability (Spearman)')
ax.set_title(f'Independence Confirmation at \u03c1={rho_test}\nRed = dependent methods, Green = independent methods')
ax.axvline(x=min(stabilities), color='gray', linestyle=':', alpha=0.5)

for bar, val in zip(bars, stabilities):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Correlation Sweep — Main Result (Paper Table 1 & Figure 3)
Full sweep \u03c1 \u2208 {0.0, 0.5, 0.7, 0.9, 0.95} with 8 methods, N_REPS=20.
This is the central experiment.

In [ ]:
# Scaling analysis: stability degradation vs rho
print('Stability degradation analysis:')
print(f'{"Method":<22} \u03c1=0.0 \u03c1=0.95   Change   % Change')
print('=' * 58)

for name in mechanism_methods:
    if name in sweep_results.get(0.0, {}) and name in sweep_results.get(0.95, {}):
        s0 = sweep_results[0.0][name]['stability']
        s95 = sweep_results[0.95][name]['stability']
        change = s95 - s0
        pct = 100 * change / s0
        print(f'{name:<22} {s0:8.4f} {s95:8.4f} {change:+8.4f} {pct:+9.1f}%')

print()
print('Key: DASH stability is flat across rho (< 0.5% variation).')
print('     LSM degrades monotonically as sequential dependency compounds.')

In [ ]:
# Equity scaling
print('Within-group equity (CV) scaling:')
print(f'{"Method":<22} \u03c1=0.0 \u03c1=0.95   Change')
print('=' * 48)

for name in mechanism_methods:
    if name in sweep_results.get(0.0, {}) and name in sweep_results.get(0.95, {}):
        e0 = sweep_results[0.0][name].get('equity_mean', float('nan'))
        e95 = sweep_results[0.95][name].get('equity_mean', float('nan'))
        print(f'{name:<22} {e0:8.4f} {e95:8.4f} {e95-e0:+8.4f}')

---
## 4. Detecting First-Mover Bias: FSI and IS Plot (Paper 5.4)

A key practical question: how can a practitioner know whether their explanations suffer
from first-mover bias? DASH's Stage 5 diagnostics address this by quantifying
explanation disagreement across the ensemble, without requiring ground truth.

In [ ]:
RHO_LEVELS = sorted(sweep_results.keys())
COLORS = {
    'Single Best': '#3498db', 'Single Best (M=200)': '#2471a3',
    'Large Single Model': '#e74c3c', 'LSM (Tuned)': '#c0392b',
    'Stochastic Retrain': '#f39c12', 'Random Selection': '#9b59b6',
    'DASH (MaxMin)': '#2ecc71', 'Naive Top-N': '#1abc9c',
}
MARKERS = {
    'Single Best': 's', 'Single Best (M=200)': 'S',
    'Large Single Model': 'X', 'LSM (Tuned)': 'x',
    'Stochastic Retrain': 'D', 'Random Selection': '^',
    'DASH (MaxMin)': 'o', 'Naive Top-N': 'v',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
method_names = list(sweep_results[RHO_LEVELS[0]].keys())

for name in method_names:
    c = COLORS.get(name, '#333')
    m = MARKERS.get(name, 'o')

    vals = [sweep_results[rho][name]['stability'] for rho in RHO_LEVELS if name in sweep_results[rho]]
    rhos = [rho for rho in RHO_LEVELS if name in sweep_results[rho]]
    axes[0].plot(rhos, vals, f'{m}-', color=c, label=name, linewidth=2, markersize=7)

    vals = [sweep_results[rho][name]['accuracy_mean'] for rho in rhos]
    axes[1].plot(rhos, vals, f'{m}-', color=c, label=name, linewidth=2, markersize=7)

    vals = [sweep_results[rho][name]['equity_mean'] for rho in rhos]
    axes[2].plot(rhos, vals, f'{m}-', color=c, label=name, linewidth=2, markersize=7)

axes[0].set_ylabel('Stability (Spearman)')
axes[0].set_title('Stability vs. Collinearity')
axes[0].legend(fontsize=7, loc='lower left')
axes[1].set_ylabel('Accuracy (Spearman vs DGP)')
axes[1].set_title('Accuracy vs. Collinearity')
axes[2].set_ylabel('Within-Group CV (lower=better)')
axes[2].set_title('Equity vs. Collinearity')
for ax in axes:
    ax.set_xlabel('Within-Group Correlation \u03c1')

fig.suptitle('DASH vs Baselines \u2014 Synthetic Linear DGP', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/correlation_sweep.pdf', bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/correlation_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/correlation_sweep.pdf')

In [ ]:
# Demonstrate diagnostics on a single synthetic run at rho=0.9
_ckpt = load_checkpoint('nb7_diagnostics_demo')
if _ckpt is not None:
    dash_demo = _ckpt['dash_demo']
    X_train = _ckpt['X_train']; y_train = _ckpt['y_train']
    X_val = _ckpt['X_val']; y_val = _ckpt['y_val']
    X_explain = _ckpt['X_explain']; X_test = _ckpt['X_test']
    groups = _ckpt['groups']
else:
    result = generate_synthetic_linear(rho=0.9, n_samples=5000, seed=SEED)
    X_train, y_train, X_val, y_val, X_explain, X_test = result[0], result[1], result[2], result[3], result[4], result[6]
    groups = np.repeat(np.arange(10), 5)

    dash_demo = DASHPipeline(
        M=M, K=K, epsilon=EPSILON, delta=DELTA, seed=SEED,
        selection_method='maxmin',
        preliminary_importance_method='gain',
    )
    dash_demo.fit(X_train, y_train, X_val, y_val, X_ref=X_explain,
                  feature_names=feature_names)

    save_checkpoint('nb7_diagnostics_demo',
                    dash_demo=dash_demo, X_train=X_train, y_train=y_train,
                    X_val=X_val, y_val=y_val, X_explain=X_explain,
                    X_test=X_test, groups=groups)

print(f'DASH fitted: {len(dash_demo.selected_indices_)} models selected from {len(dash_demo.filtered_indices_)} filtered')

In [ ]:
# IS Plot: Importance-Stability scatter with 4 quadrants
fig = dash_demo.plot_importance_stability(
    groups=groups,
    title='Importance-Stability Plot \u2014 Synthetic Linear (\u03c1=0.9)',
    annotate_top_k=10,
    figsize=(10, 8),
)
plt.show()

print('\nQuadrant interpretation:')
print('  Q1 (high imp, low FSI):  Robust Drivers \u2014 trustworthy rankings')
print('  Q2 (high imp, high FSI): Collinear Cluster Members \u2014 individual rankings unreliable')
print('  Q3 (low imp, low FSI):   Confirmed Unimportant')
print('  Q4 (low imp, high FSI):  Fragile Interactions')

In [ ]:
# FSI values for all features
diag = compute_diagnostics(dash_demo.all_shap_matrices_, feature_names=feature_names)
fsi = diag['fsi']

print('Feature Stability Index (top 15 most unstable):')
print(f'{"Feature":<15} {"FSI":>8} {"Group":>6} {"Importance":>12}')
print('=' * 45)
sorted_idx = np.argsort(fsi)[::-1]
for i in sorted_idx[:15]:
    print(f'{feature_names[i]:<15} {fsi[i]:8.4f} {groups[i]:6d} {diag["global_importance"][i]:12.4f}')

print('\nHigh FSI = feature importance varies across models = likely collinear')

In [ ]:
# Local Disagreement Map for the highest-variance observation
variance_per_obs = np.mean(dash_demo.variance_matrix_, axis=1)
high_disagreement_idx = np.argmax(variance_per_obs)

fig = local_disagreement_map(
    dash_demo.all_shap_matrices_, high_disagreement_idx,
    feature_names=feature_names,
    title=f'Local Disagreement Map (observation {high_disagreement_idx})',
)
plt.show()

---
## 5. Real-World Validation (Paper 5.5)

We validate on datasets with natural multicollinearity.

### 5a. California Housing
8 features, natural collinearity between spatial and income variables.

In [ ]:
_ckpt = load_checkpoint('nb7_california')
if _ckpt is not None:
    cal_results = _ckpt['cal_results']
else:
    from run_experiments import experiment_real_california
    cal_results = experiment_real_california()
    save_checkpoint('nb7_california', cal_results=cal_results)

print('California Housing Results:')
print(f'{"Method":<22} {"Stability":>10} {"Stability SE":>12} {"RMSE":>12} {"Ablation":>12}')
print('=' * 72)
for name, d in cal_results.items():
    if name.startswith('_'):
        continue
    se = d.get('stability_se', float('nan'))
    rmse = d.get('rmse_mean', float('nan'))
    abl = d.get('ablation_mean', float('nan'))
    print(f'{name:<22} {d["stability"]:10.4f} {se:12.4f} {rmse:12.4f} {abl:12.4f}')

if 'Single Best' in cal_results and 'DASH (MaxMin)' in cal_results:
    gap = cal_results['DASH (MaxMin)']['stability'] - cal_results['Single Best']['stability']
    print(f'\nStability improvement: +{gap:.3f}')

## 5. Real-World: Breast Cancer (Paper Table 4, Figures 4-5)
Binary classification with heavy natural collinearity (21 pairs |r|>0.9).

In [ ]:
_ckpt = load_checkpoint('nb7_breast_cancer')
if _ckpt is not None:
    bc_results = _ckpt['bc_results']
else:
    from run_experiments import experiment_real_breast_cancer
    bc_results = experiment_real_breast_cancer()
    save_checkpoint('nb7_breast_cancer', bc_results=bc_results)

print('Breast Cancer Results:')
print(f'{"Method":<22} {"Stability":>10} {"95% CI":>18} {"Ablation":>12}')
print('=' * 68)
for name, d in bc_results.items():
    if name.startswith('_'):
        continue
    ci = ''
    if 'stability_ci_lo' in d:
        ci = f"[{d['stability_ci_lo']:.3f}, {d['stability_ci_hi']:.3f}]"
    abl = d.get('ablation_mean', float('nan'))
    print(f'{name:<22} {d["stability"]:10.4f} {ci:>18} {abl:12.4f}')

if 'Single Best' in bc_results and 'DASH (MaxMin)' in bc_results:
    gap = bc_results['DASH (MaxMin)']['stability'] - bc_results['Single Best']['stability']
    print(f'\nStability improvement: +{gap:.3f}')

# Show significance tests if available
if '_significance' in bc_results:
    print('\nSignificance tests (DASH vs baselines):')
    for bl, metrics in bc_results['_significance'].items():
        for metric, vals in metrics.items():
            print(f'  vs {bl} ({metric}): p={vals["p"]:.4g}, d={vals["cohens_d"]:.3f}')

In [ ]:
# Breast Cancer diagnostics: run DASH once and show IS Plot
_ckpt = load_checkpoint('nb7_bc_diagnostics')
if _ckpt is not None:
    dash_bc = _ckpt['dash_bc']
    bc_feature_names = _ckpt['bc_feature_names']
else:
    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split

    data = load_breast_cancer()
    bc_feature_names = list(data.feature_names)
    X, y = data.data, data.target

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED)
    Xtr2, Xv, ytr2, yv = train_test_split(Xtr, ytr, test_size=0.25, random_state=SEED)

    dash_bc = DASHPipeline(
        M=M, K=K, epsilon=REAL_EPSILON, delta=DELTA, seed=SEED,
        selection_method='maxmin',
        preliminary_importance_method='gain',
        task='classification',
        epsilon_mode=REAL_EPSILON_MODE,
    )
    dash_bc.fit(Xtr2, ytr2, Xv, yv, X_ref=Xte, feature_names=bc_feature_names)
    save_checkpoint('nb7_bc_diagnostics', dash_bc=dash_bc, bc_feature_names=bc_feature_names)

print(f'Breast Cancer DASH: {len(dash_bc.selected_indices_)} models selected')

# IS Plot: Breast Cancer
fig = dash_bc.plot_importance_stability(
    title='Importance-Stability Plot \u2014 Breast Cancer Wisconsin',
    annotate_top_k=10,
    figsize=(10, 8),
)
plt.show()

print('\nRadius/perimeter/area triad should appear in Q2 (Collinear Cluster Members).')
print('Mean concave points should appear in Q1 (Robust Drivers).')

In [ ]:
# IS Plot for Breast Cancer — should identify radius/perimeter/area in Quadrant II
fig = dash_bc.plot_importance_stability(
    title='IS Plot \u2014 Breast Cancer (Binary Classification)',
    annotate_top_k=10,
    figsize=(10, 8),
)
plt.show()

# Top features by consensus importance
gi = dash_bc.global_importance_
sorted_idx = np.argsort(gi)[::-1]
print('\nTop 10 features by DASH consensus importance:')
for i in sorted_idx[:10]:
    print(f'  {bc_feature_names[i]:<25} {gi[i]:.4f}')

## 6. Real-World: Superconductor (Paper Table 4)

In [ ]:
_ckpt = load_checkpoint('nb7_superconductor')
if _ckpt is not None:
    sc_results = _ckpt['sc_results']
else:
    from run_experiments import experiment_real_superconductor
    sc_results = experiment_real_superconductor()
    save_checkpoint('nb7_superconductor', sc_results=sc_results)

print('Superconductor Results:')
print(f'{"Method":<22} {"Stability":>10} {"RMSE":>12}')
print('=' * 48)
for name, d in sc_results.items():
    rmse = d.get('rmse_mean', float('nan'))
    print(f'{name:<22} {d["stability"]:10.4f} {rmse:12.2f}')

## 7. Epsilon Sensitivity (Paper Table 5)

In [ ]:
_ckpt = load_checkpoint('nb7_epsilon')
if _ckpt is not None:
    eps_results = _ckpt['eps_results']
else:
    from run_experiments import experiment_epsilon_sensitivity
    eps_results = experiment_epsilon_sensitivity()
    save_checkpoint('nb7_epsilon', eps_results=eps_results)

print('Epsilon Sensitivity (\u03c1=0.9):')
print('\u03b5   Models Passing        K_eff  Stability   Accuracy')
print('=' * 58)

for eps_val in sorted(eps_results.keys()):
    d = eps_results[eps_val]
    # Handle either dict-of-dicts or flat dict structure
    if isinstance(d, dict) and 'DASH (MaxMin)' in d:
        d = d['DASH (MaxMin)']
    stab = d.get('stability', float('nan'))
    acc = d.get('accuracy_mean', float('nan'))
    k_eff = d.get('k_eff_mean', float('nan'))
    n_pass = d.get('n_passing', '---')
    print(f'{eps_val:6.2f} {str(n_pass):>16} {k_eff:12.1f} {stab:10.4f} {acc:10.4f}')

print('\nStability varies by < 0.001 across a 3x range of epsilon.')

## 8. Ablation Studies (Paper Figure 6)

In [ ]:
_ckpt = load_checkpoint('nb7_ablation')
if _ckpt is not None:
    abl_results = _ckpt['abl_results']
else:
    from run_experiments import experiment_ablation
    abl_results = experiment_ablation()
    save_checkpoint('nb7_ablation', abl_results=abl_results)

# experiment_ablation returns {rho: {param: {val: metrics}}}
# Display M ablation at rho=0.9
rho_key = 0.9
abl_at_rho = abl_results.get(rho_key, abl_results.get(str(rho_key), {}))

# Fallback: if abl_results has param keys directly (flat structure)
if not abl_at_rho and 'M' in abl_results:
    abl_at_rho = abl_results

if 'M' in abl_at_rho:
    print(f'Population Size (M) Ablation (rho={rho_key}):')
    print(f'{"M":>6} {"Stability":>10} {"Accuracy":>10}')
    print('=' * 30)
    for m_val in sorted(abl_at_rho['M'].keys(), key=lambda x: float(x)):
        d = abl_at_rho['M'][m_val]
        stab = d.get('stability', float('nan'))
        acc = d.get('accuracy_mean', float('nan'))
        print(f'{m_val:>6} {stab:10.4f} {acc:10.4f}')
    print('\nDiminishing returns: M=200 captures most of the benefit.')

    # Show all parameters
    for param in ['K', 'eps', 'delta']:
        if param in abl_at_rho:
            print(f'\n{param} Ablation (rho={rho_key}):')
            print(f'{param:>8} {"Stability":>10} {"Accuracy":>10}')
            print('-' * 30)
            for val in sorted(abl_at_rho[param].keys(), key=lambda x: float(x)):
                d = abl_at_rho[param][val]
                print(f'{val:>8} {d.get("stability", float("nan")):10.4f} {d.get("accuracy_mean", float("nan")):10.4f}')
else:
    print('Ablation results structure:', type(abl_results), list(abl_results.keys())[:5])

## 9. Nonlinear DGP (Paper Table 6)

In [ ]:
_ckpt = load_checkpoint('nb7_nonlinear')
if _ckpt is not None:
    nl_results = _ckpt['nl_results']
else:
    from run_experiments import experiment_nonlinear_sweep
    nl_results = experiment_nonlinear_sweep()
    save_checkpoint('nb7_nonlinear', nl_results=nl_results)

nl_rhos = sorted(nl_results.keys())

print('Nonlinear DGP: Stability and Equity')
print('\u03c1   SB Stability  DASH Stability    SB Equity   DASH Equity')
print('=' * 65)

for rho in nl_rhos:
    sb = nl_results[rho].get('Single Best', {})
    da = nl_results[rho].get('DASH (MaxMin)', {})
    sb_s = sb.get('stability', float('nan'))
    da_s = da.get('stability', float('nan'))
    sb_e = sb.get('equity_mean', float('nan'))
    da_e = da.get('equity_mean', float('nan'))
    marker = ' **' if da_s > sb_s else ''
    print(f'{rho:5.2f} {sb_s:14.4f} {da_s:15.4f} {sb_e:12.4f} {da_e:13.4f}{marker}')

print('\n** = DASH advantage. Scope boundary: \u03c1 >= 0.7 for nonlinear DGPs.')

## 10. Overlapping Correlation Structure

Tests robustness when correlation is not clean block-diagonal. Uses
chain/overlapping correlation structure at rho=0.9.

In [ ]:
_ckpt = load_checkpoint('nb7_overlapping')
if _ckpt is not None:
    ol_results = _ckpt['ol_results']
else:
    from run_experiments import experiment_overlapping
    ol_results = experiment_overlapping()
    save_checkpoint('nb7_overlapping', ol_results=ol_results)

print('Overlapping Correlation Structure Results (\u03c1=0.9):')
print(f'{"Method":<22} {"Stability":>10} {"DGP Agree":>10} {"Equity(CV)":>12} {"RMSE":>10}')
print('=' * 70)
for name, d in ol_results.items():
    stab = d.get('stability', float('nan'))
    acc = d.get('accuracy_mean', float('nan'))
    eq = d.get('equity_mean', float('nan'))
    rmse = d.get('rmse_mean', float('nan'))
    print(f'{name:<22} {stab:10.4f} {acc:10.4f} {eq:12.4f} {rmse:10.4f}')

## 11. Variance Decomposition

Separates instability into data-sampling vs model-selection variance.

> **Caveat:** `1 - stability` is a proxy for instability, not a proper variance.
> Pairwise Spearman does not yield an exact additive decomposition;
> the ratios below are indicative, not exact.

In [ ]:
_ckpt = load_checkpoint('nb7_vardecomp')
if _ckpt is not None:
    vd_results = _ckpt['vd_results']
else:
    from run_experiments import experiment_variance_decomposition
    vd_results = experiment_variance_decomposition()
    save_checkpoint('nb7_vardecomp', vd_results=vd_results)

conditions = ['data_fixed', 'model_fixed', 'both_varied']
methods = ['Single Best', 'DASH (MaxMin)']

print('Variance Decomposition (rho=0.9):')
header = 'Condition        Method               Stability'
print(header)
print('=' * len(header))
for cond in conditions:
    for m in methods:
        stab = vd_results[cond][m]['stability']
        print(f'  {cond:<16} {m:<20} {stab:.4f}')

# Instability ratios
print('\nInstability decomposition (indicative — see C5 caveat):')
for m in methods:
    total = 1.0 - vd_results['both_varied'][m]['stability']
    model = 1.0 - vd_results['data_fixed'][m]['stability']
    data = 1.0 - vd_results['model_fixed'][m]['stability']
    if total > 0:
        print(f'  {m}: model-selection={model/total:.1%}, data-sampling={data/total:.1%}')


### First-Mover Concentration Visualization (Paper Figure 2)
Shows per-feature importance within a correlated group for SB, LSM, and DASH.

In [ ]:
_ckpt = load_checkpoint('nb7_first_mover_viz')
if _ckpt is not None:
    avg_imp = _ckpt['avg_imp']
else:
    from run_experiments import experiment_first_mover_visualization
    avg_imp = experiment_first_mover_visualization()
    save_checkpoint('nb7_first_mover_viz', avg_imp=avg_imp)

# Grouped bar chart: per-feature importance within group 1
group1_features = [f'G0_f{j}' for j in range(5)]
true_imp = 2.0 / 5  # True importance per feature in group 1 (beta=2.0, 5 features)

viz_methods = ['Single Best', 'Large Single Model', 'DASH (MaxMin)']
available_viz = [m for m in viz_methods if m in avg_imp]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(5)
width = 0.25
colors_viz = {'Single Best': '#95a5a6', 'Large Single Model': '#e74c3c', 'DASH (MaxMin)': '#2ecc71'}

for i, name in enumerate(available_viz):
    imp = avg_imp[name][:5]  # First 5 features = group 1
    ax.bar(x + i * width, imp, width, label=name, color=colors_viz.get(name, 'gray'),
           edgecolor='white')

ax.axhline(y=true_imp, color='black', linestyle='--', linewidth=1.5,
           label=f'True importance ({true_imp:.2f})', alpha=0.7)
ax.set_xlabel('Feature within Correlated Group 1')
ax.set_ylabel('Mean |SHAP| Importance')
ax.set_title('First-Mover Bias: Importance Concentration Within a Correlated Group\n'
             f'(\u03c1=0.9, true importance = {true_imp:.2f} per feature)')
ax.set_xticks(x + width)
ax.set_xticklabels(group1_features)
ax.legend()
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
fig.savefig('results/figures/first_mover_concentration.pdf', bbox_inches='tight', dpi=300)
fig.savefig('results/figures/first_mover_concentration.png', bbox_inches='tight', dpi=150)
print('Saved to results/figures/first_mover_concentration.{pdf,png}')
plt.show()

# Concentration metric
for name in available_viz:
    imp = avg_imp[name][:5]
    concentration = np.max(imp) / np.sum(imp) if np.sum(imp) > 0 else 0
    print(f'{name}: max/sum = {concentration:.3f} (perfect equity = {1/5:.3f})')

---
## 12. First-Mover Bias Isolation (IMPL_PLAN B3)

Shows concentration growing with tree count for a single sequential model,
while an independent ensemble of the same total compute remains equitable.
This is the direct mechanistic evidence that first-mover bias accumulates
with depth.

In [ ]:
_ckpt = load_checkpoint('nb7_first_mover_bias')
if _ckpt is not None:
    fmb_results = _ckpt['fmb_results']
else:
    from run_experiments import experiment_first_mover_bias
    fmb_results = experiment_first_mover_bias()
    save_checkpoint('nb7_first_mover_bias', fmb_results=fmb_results)

# Table
print('First-Mover Bias Isolation:')
print(f'{"n_estimators":>14} {"Single Conc":>14} {"Indep Conc":>14} {"Ratio":>8}')
print('=' * 55)
for n_est in sorted(fmb_results.keys(), key=int):
    d = fmb_results[n_est]
    sc = d['single_concentration']
    dc = d['independent_concentration']
    print(f'{n_est:>14} {sc:>14.4f} {dc:>14.4f} {sc/dc:>8.2f}')

# Line plot with error bars
xs = sorted(int(k) for k in fmb_results.keys())
sc = [fmb_results[str(n)]['single_concentration'] for n in xs]
dc = [fmb_results[str(n)]['independent_concentration'] for n in xs]
sc_err = [fmb_results[str(n)].get('single_concentration_std', 0) for n in xs]
dc_err = [fmb_results[str(n)].get('independent_concentration_std', 0) for n in xs]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(xs, sc, yerr=sc_err, fmt='s-', color='#e74c3c',
            label='Single Sequential Model', linewidth=2, capsize=4, markersize=7)
ax.errorbar(xs, dc, yerr=dc_err, fmt='o-', color='#2ecc71',
            label='Independent Ensemble', linewidth=2, capsize=4, markersize=7)
ax.axhline(y=0.2, color='black', linestyle='--', alpha=0.5, label='Perfect equity (1/5)')
ax.set_xlabel('Number of Trees (n_estimators)')
ax.set_ylabel('Concentration (max/sum within group)')
ax.set_title('First-Mover Bias Isolation: Concentration Grows with Tree Count')
ax.set_xscale('log')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 13. Statistical Significance Tests (Paper 5.2)

Wilcoxon signed-rank tests with Holm-Bonferroni correction.
Key comparisons: DASH vs SB (should be significant at high rho),
DASH vs SR (should be NOT significant — the independence principle).

In [ ]:
# Pairwise significance tests from sweep results with Holm-Bonferroni correction
comparisons = [
    (0.7, 'DASH (MaxMin)', 'Single Best'),
    (0.9, 'DASH (MaxMin)', 'Single Best'),
    (0.9, 'DASH (MaxMin)', 'Large Single Model'),
    (0.9, 'DASH (MaxMin)', 'Stochastic Retrain'),
    (0.95, 'DASH (MaxMin)', 'Single Best'),
    (0.95, 'DASH (MaxMin)', 'Large Single Model'),
]

# Collect all raw p-values first
raw_tests = []
for rho, m1, m2 in comparisons:
    if rho not in sweep_results:
        continue
    d1 = sweep_results[rho].get(m1, {})
    d2 = sweep_results[rho].get(m2, {})

    for metric_key, metric_name in [('acc_runs', 'Accuracy'), ('eq_runs', 'Equity')]:
        r1 = d1.get(metric_key)
        r2 = d2.get(metric_key)
        if r1 is None or r2 is None:
            continue
        r1, r2 = np.array(r1), np.array(r2)
        if len(r1) != len(r2) or len(r1) < 5:
            continue
        try:
            stat, p = wilcoxon(r1, r2)
        except Exception:
            continue
        diff = r1 - r2
        d_val = np.mean(diff) / (np.std(diff) + 1e-12)
        raw_tests.append({
            'rho': rho, 'm1': m1, 'm2': m2,
            'metric': metric_name, 'p_raw': p, 'cohens_d': d_val,
        })

# Apply Holm-Bonferroni correction
raw_pvals = [t['p_raw'] for t in raw_tests]
if raw_pvals:
    corrected = holm_bonferroni(raw_pvals)
    for t, p_adj in zip(raw_tests, corrected):
        t['p_adjusted'] = p_adj

print('Statistical Significance Tests (Wilcoxon signed-rank, Holm-Bonferroni corrected):')
print(f'\u03c1 {"Comparison":<35} {"Metric":<12}    p (raw)    p (adj)    Cohen d')
print('=' * 90)

for t in raw_tests:
    sig = '***' if t['p_adjusted'] < 0.001 else '**' if t['p_adjusted'] < 0.01 else '*' if t['p_adjusted'] < 0.05 else 'n.s.'
    label = f'{t["m1"]} vs {t["m2"]}'
    print(f'{t["rho"]:5.2f} {label:<35} {t["metric"]:<12} {t["p_raw"]:10.4f} {t["p_adjusted"]:10.4f} {t["cohens_d"]:+10.2f}  {sig}')

print(f'\n{len(raw_tests)} tests total. '
      f'{sum(1 for t in raw_tests if t["p_adjusted"] < 0.05)}/{len(raw_tests)} significant after correction.')

## 14. Computational Cost Summary (Paper Table)

In [ ]:
# Timing table from sweep at rho=0.9
from run_experiments import format_timing_table

format_timing_table(sweep_results, rho=0.9)

## 15. Success Criteria

In [ ]:
print('\n' + '='*70)
print('SUCCESS CRITERIA')
print('='*70)
criteria = []

# C1: Stability wins (linear) — DASH > SB on >=4/5 rho levels
wins = sum(1 for rho in sweep_results
           if 'DASH (MaxMin)' in sweep_results[rho] and 'Single Best' in sweep_results[rho]
           and sweep_results[rho]['DASH (MaxMin)']['stability'] > sweep_results[rho]['Single Best']['stability'])
total_rho = sum(1 for rho in sweep_results
                if 'DASH (MaxMin)' in sweep_results[rho] and 'Single Best' in sweep_results[rho])
passed = wins >= 4
criteria.append(('Stability wins (linear)', f'{wins}/{total_rho}', passed))
print(f'1. Stability wins: {wins}/{total_rho} \u2192 {"PASS" if passed else "FAIL"}')

# C2: Accuracy at rho=0.9
if 0.9 in sweep_results and 'DASH (MaxMin)' in sweep_results[0.9]:
    acc = sweep_results[0.9]['DASH (MaxMin)']['accuracy_mean']
    passed = acc >= 0.90
    criteria.append(('Accuracy at \u03c1=0.9', f'{acc:.4f}', passed))
    print(f'2. Accuracy at \u03c1=0.9: {acc:.4f} \u2192 {"PASS" if passed else "FAIL"}')

# C3: Safety at rho=0
if 0.0 in sweep_results and 'DASH (MaxMin)' in sweep_results[0.0] and 'Single Best' in sweep_results[0.0]:
    gap = abs(sweep_results[0.0]['DASH (MaxMin)']['stability'] - sweep_results[0.0]['Single Best']['stability'])
    passed = gap < 0.1
    criteria.append(('Safety at \u03c1=0', f'gap={gap:.4f}', passed))
    print(f'3. Safety at \u03c1=0: gap={gap:.4f} \u2192 {"PASS" if passed else "FAIL"}')

# C4: BC stability
if bc_results and 'DASH (MaxMin)' in bc_results:
    bc_stab = bc_results['DASH (MaxMin)']['stability']
    passed = bc_stab > 0.80
    criteria.append(('Breast Cancer stab>0.80', f'{bc_stab:.4f}', passed))
    print(f'4. Breast Cancer: {bc_stab:.4f} \u2192 {"PASS" if passed else "FAIL"}')

# Summary
n_pass = sum(1 for _, _, p in criteria if p)
print(f'\nOverall: {n_pass}/{len(criteria)} criteria passed')

elapsed_total = time.time() - _notebook_start
print(f'\nTotal notebook runtime: {elapsed_total/60:.1f} min')